#Proyecto: Agente Inteligente para Clínica Salud

##Objetivo final

Construir un asistente inteligente basado en IA capaz de responder consultas de pacientes utilizando exclusivamente la documentación oficial de la clínica mediante una arquitectura RAG (Retrieval-Augmented Generation).

# 1.- Diseño y Planificación del Proyecto

##Este proyecto propone el desarrollo de un Agente Inteligente basado en Inteligencia Artificial Generativa y Recuperación Aumentada por Recuperación (RAG), capaz de responder consultas utilizando exclusivamente la documentación oficial de la clínica, garantizando respuestas coherentes, consistentes y alineadas con la información institucional.

##El agente actuará como un asistente virtual disponible las 24 horas del día para orientar a pacientes y usuarios, sin reemplazar el criterio médico ni emitir diagnósticos.

# Problema

### Actualmente, los pacientes enfrentan diversas dificultades al intentar obtener información sobre los servicios de una clínica, entre ellas:

### Saturación de líneas telefónicas.


*   Largos tiempos de espera.
*   Información distribuida en múltiples documentos.
*   Respuestas inconsistentes entre distintos canales de atención.
*   Dependencia del horario de funcionamiento del personal administrativo.


### Estas situaciones afectan la experiencia del paciente y aumentan la carga de trabajo del personal de recepción.

In [52]:
# ==========================================
# INSTALACIÓN LIBRERÍAS RAG
# ==========================================


!pip install -q \
langchain \
langchain-community \
langchain-classic \
langchain-text-splitters \
pypdf \
faiss-cpu \
cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 8.5 MB/s eta 0:00:00


In [53]:
import pypdf
import faiss
import sentence_transformers
import cohere

print("Entorno RAG listo correctamente")

Entorno RAG listo correctamente


In [54]:
# ==========================================
# CREACIÓN DE CARPETAS
# ==========================================

from pathlib import Path


carpetas = [

    "documentos_clinica",

    "base_vectorial",

    "embeddings",

    "modelo",

    "resultados"

]


for carpeta in carpetas:

    Path(carpeta).mkdir(
        exist_ok=True
    )

print("Carpetas creadas correctamente")

print(
    "✅ Estructura creada correctamente"
)

Carpetas creadas correctamente
✅ Estructura creada correctamente


In [55]:
# ==========================================
# CONFIGURACIÓN GENERAL
# ==========================================


CONFIG = {

    "proyecto":
    "Agente IA Clínica Horizonte",

    "version":
    "1.0",

    "tipo":
    "RAG Atención Paciente",

    "documentos":
    5,

    "vector_database":
    "FAISS",

    "embedding":
    "all-MiniLM-L6-v2",

    "chunk_size":
    3000,

    "chunk_overlap":
    300

}


CONFIG

{'proyecto': 'Agente IA Clínica Horizonte',
 'version': '1.0',
 'tipo': 'RAG Atención Paciente',
 'documentos': 5,
 'vector_database': 'FAISS',
 'embedding': 'all-MiniLM-L6-v2',
 'chunk_size': 3000,
 'chunk_overlap': 300}

##2.- Base Documental

###Creación y carga de documentos PDF clínicos

In [56]:
# ==========================================
# CREACIÓN DE DOCUMENTOS DE PRUEBA
# ==========================================


from pathlib import Path


documentos = {

"politica_privacidad_datos_paciente.txt":

"""
POLÍTICA DE PRIVACIDAD DE DATOS DEL PACIENTE

La clínica protege la información personal
y médica de todos sus pacientes.

Los datos solo serán utilizados para fines
de atención médica, gestión administrativa
y cumplimiento normativo.

El acceso a la información está limitado
a personal autorizado.
""",


"preguntas_frecuentes_consultas_turnos.txt":

"""
PREGUNTAS FRECUENTES SOBRE CONSULTAS Y TURNOS

Los pacientes pueden solicitar horas médicas
por teléfono, plataforma web o recepción.

Se recomienda llegar 15 minutos antes
de la hora programada.

La confirmación del turno será enviada
por los canales registrados.
""",


"politica_cancelaciones_reagendamiento.txt":

"""
POLÍTICA DE CANCELACIONES Y REAGENDAMIENTO

Los pacientes pueden modificar sus citas
informando previamente a la clínica.

Las solicitudes de cambio están sujetas
a disponibilidad de agenda.

La inasistencia sin aviso puede afectar
futuras reservas.
""",


"guia_convenios_coberturas_medicas.txt":

"""
GUÍA DE CONVENIOS Y COBERTURAS MÉDICAS

La clínica trabaja con diferentes convenios
y seguros médicos.

El paciente debe presentar documentación
de identificación y cobertura vigente.
""",


"instrucciones_pre_post_consulta.txt":

"""
INSTRUCCIONES PRE Y POST CONSULTA

Antes de la consulta:

- Presentar identificación.
- Llevar antecedentes médicos.
- Cumplir indicaciones previas.


Después de la consulta:

- Seguir indicaciones del profesional.
- Realizar controles programados.
- Consultar ante síntomas de alerta.
"""

}


ruta = Path("documentos_clinica")


for nombre, contenido in documentos.items():

    archivo = ruta / nombre

    archivo.write_text(
        contenido,
        encoding="utf-8"
    )


print("✅ Documentos creados")

✅ Documentos creados


In [57]:
# ==========================================
# LECTURA DOCUMENTAL
# ==========================================


documentos_cargados = []


for archivo in Path(
    "documentos_clinica"
).glob("*.txt"):


    texto = archivo.read_text(
        encoding="utf-8"
    )


    documentos_cargados.append({

        "contenido": texto,

        "fuente": archivo.name

    })


print(
    "Documentos cargados:",
    len(documentos_cargados)
)

Documentos cargados: 5


In [58]:
# ==========================================
# PRIMER DOCUMENTO
# ==========================================


print(
    documentos_cargados[0]["fuente"]
)


print(
    documentos_cargados[0]["contenido"]
)

instrucciones_pre_post_consulta.txt

INSTRUCCIONES PRE Y POST CONSULTA

Antes de la consulta:

- Presentar identificación.
- Llevar antecedentes médicos.
- Cumplir indicaciones previas.


Después de la consulta:

- Seguir indicaciones del profesional.
- Realizar controles programados.
- Consultar ante síntomas de alerta.



##División de texto y preparación del conocimiento

In [59]:
# ==========================================
# IMPORTAR HERRAMIENTAS
# ==========================================


from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter


print(
    "✅ Herramientas cargadas"
)

✅ Herramientas cargadas


In [60]:
# ==========================================
# CREAR DOCUMENTOS LANGCHAIN
# ==========================================


documentos_langchain = []


for documento in documentos_cargados:


    doc = Document(

        page_content=documento["contenido"],

        metadata={

            "source":
            documento["fuente"],

            "tipo":
            "documento_paciente"

        }

    )


    documentos_langchain.append(doc)



print(
    "Documentos preparados:",
    len(documentos_langchain)
)

Documentos preparados: 5


In [61]:
chunk_size = 1000
chunk_overlap = 100

In [62]:
# ==========================================
# CONFIGURACIÓN CHUNKS
# ==========================================


splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=100,

    length_function=len

)


print(
    "✅ Fragmentador configurado"
)

✅ Fragmentador configurado


In [63]:
# ==========================================
# CREAR FRAGMENTOS
# ==========================================


fragmentos = splitter.split_documents(

    documentos_langchain

)


print(
    "Cantidad de fragmentos:",
    len(fragmentos)
)

Cantidad de fragmentos: 5


In [64]:
# ==========================================
# REVISAR FRAGMENTOS
# ==========================================


for i, fragmento in enumerate(fragmentos):

    print(
        "Fragmento:",
        i+1
    )

    print(
        "Fuente:",
        fragmento.metadata["source"]
    )

    print(
        "Tamaño:",
        len(fragmento.page_content),
        "caracteres"
    )

    print("-"*50)

Fragmento: 1
Fuente: instrucciones_pre_post_consulta.txt
Tamaño: 285 caracteres
--------------------------------------------------
Fragmento: 2
Fuente: guia_convenios_coberturas_medicas.txt
Tamaño: 183 caracteres
--------------------------------------------------
Fragmento: 3
Fuente: politica_privacidad_datos_paciente.txt
Tamaño: 299 caracteres
--------------------------------------------------
Fragmento: 4
Fuente: politica_cancelaciones_reagendamiento.txt
Tamaño: 249 caracteres
--------------------------------------------------
Fragmento: 5
Fuente: preguntas_frecuentes_consultas_turnos.txt
Tamaño: 264 caracteres
--------------------------------------------------


In [65]:
# ==========================================
# MOSTRAR CONTENIDO
# ==========================================


print(
    fragmentos[0].page_content
)

INSTRUCCIONES PRE Y POST CONSULTA

Antes de la consulta:

- Presentar identificación.
- Llevar antecedentes médicos.
- Cumplir indicaciones previas.


Después de la consulta:

- Seguir indicaciones del profesional.
- Realizar controles programados.
- Consultar ante síntomas de alerta.


In [66]:
# ==========================================
# VALIDAR METADATA
# ==========================================


print(
    fragmentos[0].metadata
)

{'source': 'instrucciones_pre_post_consulta.txt', 'tipo': 'documento_paciente'}


In [67]:
# ==========================================
# GUARDAR FRAGMENTOS
# ==========================================


import pickle


with open(
    "fragmentos_clinica.pkl",
    "wb"
) as archivo:


    pickle.dump(

        fragmentos,

        archivo

    )


print(
    "✅ Fragmentos guardados"
)

✅ Fragmentos guardados


#Creación de Embeddings y Memoria Inteligente FAISS

In [68]:
!pip install -U langchain-huggingface -q

In [69]:
# ==========================================
# IMPORTAR EMBEDDINGS
# ==========================================


from langchain_community.embeddings import HuggingFaceEmbeddings


print(
    "✅ Librería embeddings cargada"
)

✅ Librería embeddings cargada


In [70]:
# ==========================================
# MODELO EMBEDDING
# ==========================================


modelo_embeddings = HuggingFaceEmbeddings(

    model_name=
    "sentence-transformers/all-MiniLM-L6-v2"

)


print(
    "✅ Modelo de embeddings listo"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Modelo de embeddings listo


In [71]:
# ==========================================
# PRUEBA DE VECTOR
# ==========================================


texto_prueba = [

    "¿Cómo puedo cancelar mi consulta médica?"

]


vector = modelo_embeddings.embed_documents(

    texto_prueba

)


print(
    "Dimensión del vector:",
    len(vector[0])
)

Dimensión del vector: 384


In [72]:
# ==========================================
# IMPORTAR FAISS
# ==========================================


from langchain_community.vectorstores import FAISS


print(
    "✅ FAISS cargado"
)

✅ FAISS cargado


In [73]:
# ==========================================
# CREAR BASE VECTORIAL
# ==========================================


base_vectorial = FAISS.from_documents(

    fragmentos,

    modelo_embeddings

)


print(
    "✅ Base vectorial creada correctamente"
)

✅ Base vectorial creada correctamente


In [74]:
# ==========================================
# GUARDAR FAISS
# ==========================================


base_vectorial.save_local(

    "base_vectorial_clinica"

)


print(
    "✅ Memoria clínica guardada"
)

✅ Memoria clínica guardada


In [75]:
consulta = """

¿Cómo puedo cambiar una hora médica?

"""

In [76]:
# ==========================================
# BUSQUEDA SEMANTICA
# ==========================================


resultados = base_vectorial.similarity_search(

    consulta,

    k=3

)


for resultado in resultados:


    print(
        "Fuente:",
        resultado.metadata["source"]
    )


    print(
        resultado.page_content[:300]
    )


    print("-"*50)

Fuente: politica_cancelaciones_reagendamiento.txt
POLÍTICA DE CANCELACIONES Y REAGENDAMIENTO

Los pacientes pueden modificar sus citas
informando previamente a la clínica.

Las solicitudes de cambio están sujetas
a disponibilidad de agenda.

La inasistencia sin aviso puede afectar
futuras reservas.
--------------------------------------------------
Fuente: politica_privacidad_datos_paciente.txt
POLÍTICA DE PRIVACIDAD DE DATOS DEL PACIENTE

La clínica protege la información personal
y médica de todos sus pacientes.

Los datos solo serán utilizados para fines
de atención médica, gestión administrativa
y cumplimiento normativo.

El acceso a la información está limitado
a personal autorizado.
--------------------------------------------------
Fuente: preguntas_frecuentes_consultas_turnos.txt
PREGUNTAS FRECUENTES SOBRE CONSULTAS Y TURNOS

Los pacientes pueden solicitar horas médicas
por teléfono, plataforma web o recepción.

Se recomienda llegar 15 minutos antes
de la hora programada.

La c

In [ ]:
# ==========================================
# PRUEBAS DE MEMORIA
# ==========================================


preguntas = [

"¿Cómo protege la clínica mis datos?",

"¿Qué debo hacer antes de mi consulta?",

"¿Qué pasa si necesito cambiar mi turno?",

"¿Qué seguros acepta la clínica?"

]


for pregunta in preguntas:


    print("\nPregunta:")
    print(pregunta)


    respuesta = base_vectorial.similarity_search(

        pregunta,

        k=1

    )


    print(

        "Documento encontrado:",

        respuesta[0].metadata["source"]

    )

#Creación del Retriever y Motor de Búsqueda del Paciente

In [77]:
# ==========================================
# CREAR RETRIEVER CLÍNICO
# ==========================================


retriever = base_vectorial.as_retriever(

    search_kwargs={

        "k": 3

    }

)


print(
    "✅ Retriever creado correctamente"
)

✅ Retriever creado correctamente


In [79]:
# ==========================================
# PRUEBA DEL RETRIEVER
# ==========================================


pregunta = """

¿Qué debo hacer antes de mi consulta?

"""


documentos_encontrados = retriever.invoke(

    pregunta

)


print(

    "Documentos encontrados:",

    len(documentos_encontrados)

)

Documentos encontrados: 3


In [80]:
# ==========================================
# MOSTRAR RESULTADOS
# ==========================================


for i, documento in enumerate(documentos_encontrados):


    print(
        "Resultado:",
        i+1
    )


    print(
        "Fuente:",
        documento.metadata["source"]
    )


    print(
        documento.page_content[:400]
    )


    print("-"*60)

Resultado: 1
Fuente: instrucciones_pre_post_consulta.txt
INSTRUCCIONES PRE Y POST CONSULTA

Antes de la consulta:

- Presentar identificación.
- Llevar antecedentes médicos.
- Cumplir indicaciones previas.


Después de la consulta:

- Seguir indicaciones del profesional.
- Realizar controles programados.
- Consultar ante síntomas de alerta.
------------------------------------------------------------
Resultado: 2
Fuente: politica_cancelaciones_reagendamiento.txt
POLÍTICA DE CANCELACIONES Y REAGENDAMIENTO

Los pacientes pueden modificar sus citas
informando previamente a la clínica.

Las solicitudes de cambio están sujetas
a disponibilidad de agenda.

La inasistencia sin aviso puede afectar
futuras reservas.
------------------------------------------------------------
Resultado: 3
Fuente: preguntas_frecuentes_consultas_turnos.txt
PREGUNTAS FRECUENTES SOBRE CONSULTAS Y TURNOS

Los pacientes pueden solicitar horas médicas
por teléfono, plataforma web o recepción.

Se recomienda llegar 15 

In [81]:
# ==========================================
# FUNCIÓN BUSCADOR CLÍNICO
# ==========================================


def buscar_documentos(pregunta):


    resultados = retriever.invoke(

        pregunta

    )


    return resultados



print(
    "✅ Función creada"
)

✅ Función creada


In [82]:
# ==========================================
# CASOS DE PRUEBA
# ==========================================


preguntas = [

    "¿Cómo protegen mis datos médicos?",

    "Necesito cambiar mi hora médica",

    "¿Qué documentos necesito para usar mi seguro?",

    "¿Qué debo hacer después de la consulta?"

]


for pregunta in preguntas:


    print("\nPaciente:")
    print(pregunta)


    resultados = buscar_documentos(
        pregunta
    )


    print(
        "Documento encontrado:"
    )


    for resultado in resultados[:2]:

        print(
            "-",
            resultado.metadata["source"]
        )


Paciente:
¿Cómo protegen mis datos médicos?
Documento encontrado:
- politica_privacidad_datos_paciente.txt
- instrucciones_pre_post_consulta.txt

Paciente:
Necesito cambiar mi hora médica
Documento encontrado:
- preguntas_frecuentes_consultas_turnos.txt
- politica_cancelaciones_reagendamiento.txt

Paciente:
¿Qué documentos necesito para usar mi seguro?
Documento encontrado:
- instrucciones_pre_post_consulta.txt
- politica_privacidad_datos_paciente.txt

Paciente:
¿Qué debo hacer después de la consulta?
Documento encontrado:
- instrucciones_pre_post_consulta.txt
- politica_cancelaciones_reagendamiento.txt


In [83]:
# ==========================================
# MOSTRAR FUENTES
# ==========================================


def mostrar_fuentes(pregunta):


    resultados = retriever.invoke(
        pregunta
    )


    fuentes = []


    for documento in resultados:

        fuentes.append(
            documento.metadata["source"]
        )


    return fuentes



mostrar_fuentes(
    "¿Puedo cancelar una consulta?"
)

['politica_cancelaciones_reagendamiento.txt',
 'instrucciones_pre_post_consulta.txt',
 'preguntas_frecuentes_consultas_turnos.txt']

#Construcción del Agente RAG Conversacional

In [84]:
# ==========================================
# INSTALAR COHERE
# ==========================================


!pip install -q langchain-cohere cohere


print(
    "✅ Cliente Cohere instalado"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 15.0 MB/s eta 0:00:00
✅ Cliente Cohere instalado


In [ ]:
# ==========================================
# CONFIGURAR API KEY
# ==========================================


import os

from getpass import getpass


os.environ["COHERE_API_KEY"] = getpass(
    "Ingrese su API Key de Cohere: "
)


print(
    "✅ API configurada"
)

In [ ]:
# ==========================================
# CREAR MODELO IA
# ==========================================


from langchain_cohere import ChatCohere


llm = ChatCohere(

    model="command-a-03-2025",

    temperature=0

)


print(
    "✅ Modelo IA conectado"
)

In [ ]:
# ==========================================
# PROMPT DEL AGENTE
# ==========================================


from langchain_core.prompts import PromptTemplate



prompt_clinica = PromptTemplate(

    input_variables=[
        "context",
        "question"
    ],


    template="""


Eres el asistente virtual
de Clínica Inteligente Horizonte.


Tu función es ayudar a pacientes
con información administrativa.


Utiliza únicamente el contexto entregado.


Si la información no está disponible,
indica claramente:

"No encuentro esa información
en los documentos disponibles."


Contexto:

{context}


Pregunta del paciente:

{question}


Respuesta:


"""

)


print(
    "✅ Prompt clínico creado"
)

In [ ]:
# ==========================================
# CREAR AGENTE RAG
# ==========================================


from langchain_classic.chains import RetrievalQA



agente_clinico = RetrievalQA.from_chain_type(

    llm=llm,

    retriever=retriever,

    return_source_documents=True,

    chain_type_kwargs={

        "prompt":
        prompt_clinica

    }

)



print(
    "✅ Agente RAG creado"
)

In [ ]:
# ==========================================
# PRUEBA 1
# ==========================================


pregunta = """

¿Cómo puedo cambiar mi hora médica?

"""


respuesta = agente_clinico.invoke(

    {
        "query":
        pregunta
    }

)


print(
    respuesta["result"]
)

In [ ]:
# ==========================================
# MOSTRAR FUENTES
# ==========================================


for documento in respuesta["source_documents"]:

    print(
        documento.metadata["source"]
    )

In [ ]:
# ==========================================
# PRUEBAS FINALES
# ==========================================


preguntas = [

"¿Cómo protege la clínica mis datos?",

"¿Qué debo llevar a mi consulta?",

"¿Puedo usar mi seguro médico?",

"¿Qué pasa si no puedo asistir?"

]


for pregunta in preguntas:


    resultado = agente_clinico.invoke(

        {
            "query":
            pregunta
        }

    )


    print("="*70)

    print(
        "Paciente:"
    )

    print(
        pregunta
    )


    print(
        "\nAsistente:"
    )

    print(
        resultado["result"]
    )

In [ ]:
# ==========================================
# CONFIGURACIÓN FINAL
# ==========================================


CONFIG_FINAL = {

    "Proyecto":
    "Agente IA Clínica Horizonte",


    "Tipo":
    "RAG Atención Paciente",


    "Documentos":
    5,


    "Vector_DB":
    "FAISS",


    "Embeddings":
    "all-MiniLM-L6-v2",


    "LLM":
    "Cohere Command"


}


CONFIG_FINAL